### Download de date pentru inserat in tabele

In [ ]:
%pip install yfinance
%pip install matplotlib
%pip install pandas
%pip install numpy
%pip install oracledb

In [ ]:
# import modules
from datetime import datetime
import yfinance as yf
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import oracledb
import typing
import re

In [ ]:
# Metode pentru extras date de la bursa
class StockPrice:
   ticker: str
   id_bursa: str | None
   data_cotatie: str # of the form YYYY-MM-DD HH:MM:SS
   pret_deschidere: np.float64
   pret_inchidere: np.float64
   pret_maxim: np.float64
   pret_minim: np.float64
   volum: np.float64
   
   def __init__(self, ticker: str, id_bursa: str | None, data_cotatie: str, pret_deschidere: np.float64, pret_inchidere: np.float64, pret_maxim: np.float64, pret_minim: np.float64, volum: np.float64):
      # save ticker without exchange suffix (e.g. TLV.RO -> TLV)
      self.ticker = ticker.rsplit('.', 1)[0]
      self.id_bursa = id_bursa
      self.data_cotatie = data_cotatie
      self.pret_deschidere = pret_deschidere
      self.pret_inchidere = pret_inchidere
      self.pret_maxim = pret_maxim
      self.pret_minim = pret_minim
      self.volum = volum

   def __str__(self):
      return f"StockPrice(ticker={self.ticker}, id_bursa={self.id_bursa}, data_cotatie={self.data_cotatie}, pret_deschidere={self.pret_deschidere}, pret_inchidere={self.pret_inchidere}, pret_maxim={self.pret_maxim}, pret_minim={self.pret_minim}, volum={self.volum})"


def conv_yf_to_stock_price(yf_data: pd.DataFrame, ticker: str, id_bursa: str | None) -> list[StockPrice]:
   """
   Input va fi de forma: MultiIndex([
      ( 'Close', 'NVDA'),
      (  'High', 'NVDA'),
      (   'Low', 'NVDA'),
      (  'Open', 'NVDA'),
      ('Volume', 'NVDA')],
      names=['Price', 'Ticker'])
   Coloana de index va fi timestamp-ul
   
   Returneaza un array de obiecte de tip StockPrice, cate unul pentru fiecare rand din yf_data
   """
   
   stock_prices = []
   for index, row in yf_data.iterrows():
      # type of index is actually pd.Timestamp
      idx_timestamp: pd.Timestamp = index # type: ignore 
      data_cotatie = idx_timestamp.strftime('%Y-%m-%d %H:%M:%S')
      
      pret_deschidere = row['Open'].values[0]
      if pret_deschidere is None:
         raise ValueError(f"Open price is None for date {data_cotatie}")
      
      pret_inchidere = row['Close'].values[0]
      if pret_inchidere is None:
         raise ValueError(f"Close price is None for date {data_cotatie}")
      
      pret_maxim = row['High'].values[0]
      if pret_maxim is None:
         raise ValueError(f"High price is None for date {data_cotatie}")
      
      pret_minim = row['Low'].values[0]
      if pret_minim is None:
         raise ValueError(f"Low price is None for date {data_cotatie}")
      
      volum = row['Volume'].values[0]
      if volum is None:
         raise ValueError(f"Volume is None for date {data_cotatie}")
      
      stock_price = StockPrice(ticker, id_bursa, data_cotatie, pret_deschidere, pret_inchidere, pret_maxim, pret_minim, volum)
      stock_prices.append(stock_price)
   return stock_prices


def download_ticker(ticker: str, id_bursa: str | None, start_date: str, end_date: str, interval: str = '1d', show_graph: bool = False) -> list[StockPrice]:
   """
   Download the stock price data for the given ticker and date range using yfinance.
   id_bursa is explicitly provided (e.g. NYSE, NASDAQ, MEXI, AEB, BVB).
   Return a list of StockPrice objects.
   """
   yf_data: pd.DataFrame | None = yf.download(
      tickers=ticker, 
      start=start_date, 
      end=end_date, 
      interval=interval)
   if yf_data is None:
      raise ValueError(f"Failed to download data for ticker {ticker}")
   
   if show_graph:
      plt.figure(figsize=(10, 5))
      plt.plot(yf_data.index, yf_data['Close'])
      plt.title(f'{ticker} Stock Price')
      plt.xlabel('Date')
      plt.ylabel('Close Price')
      plt.grid()
      plt.show()
   
   return conv_yf_to_stock_price(yf_data, ticker, id_bursa)


In [ ]:
# # Ex de utilizare
# stock_prices = download_ticker(
#    ticker='TLV.RO',
#    id_bursa='BVB',
#    start_date='2025-01-01',
#    end_date=datetime.now().strftime('%Y-%m-%d'),
#    interval='1wk', show_graph=True)
# len(stock_prices[:25])


In [ ]:
stock_prices = download_ticker(ticker='ALV.DE', id_bursa='FSE', 
   start_date='2025-06-01', end_date=datetime.now().strftime('%Y-%m-%d'), 
   interval='1wk', show_graph=True)
len(stock_prices)

In [ ]:
# DB Stuff
ORACLE_HOST = "10.19.49.10"
ORACLE_PORT = 1522
ORACLE_SERVICE = "XE"
ORACLE_USER = "proiect"
ORACLE_PASSWORD = "proiect"

def db_get_connection() -> oracledb.Connection:
   """Se conecteaza la baza de date si returneaza conexiunea

   Returns:
      oracledb.Connection: Conexiunea la baza de date Oracle
   """
   dsn = f"{ORACLE_HOST}:{ORACLE_PORT}/{ORACLE_SERVICE}"
   print(f"Connecting to Oracle DB with DSN: '{dsn}'")
   connection: oracledb.Connection = oracledb.connect(user=ORACLE_USER, password=ORACLE_PASSWORD, dsn=dsn)
   print(f"Connected to Oracle DB with DSN: '{dsn}'; user: '{ORACLE_USER}'")
   return connection

def db_run_sql(conn: oracledb.Connection, sql: str, params: typing.Optional[dict] = None, fetch: bool = False, commit: bool = False) -> typing.Optional[list]: 
   """Executa o interogare SQL pe baza de date

   Args:
      conn (oracledb.Connection): Conexiunea la baza de date
      sql (str): Interogarea SQL de executat
      params (dict, optional): Parametrii pentru interogare. Defaults to None.
      fetch (bool, optional): Daca True, returneaza rezultatele interogarii. Defaults to False.

   Returns:
      list: Rezultatele interogarii daca fetch este True, altfel None
   """
   sql_print = str(sql).replace('\n', ' ').replace('\t', ' ')
   if len(sql_print) > 100:
      sql_print = sql_print[:100] + '...'
   print(f"Running SQL query: fetch={fetch}, commit={commit}, SQL: \"{sql_print}\", params: {params}")
   cursor = conn.cursor()
   if params:
      cursor.execute(sql, params)
   else:
      cursor.execute(sql)
   
   if fetch:
      return cursor.fetchall()

   if commit:
      conn.commit()
   return None

oracle_conn: oracledb.Connection = db_get_connection()

In [ ]:

# Creare date de inserare pentru 02-phase2.sql: 
# Tabel 1: simbol_bursier -> 5 elemente
#  - ticker: # 
#  - id_companie: COMPANIE.ID_COMPANIE
#  - id_bursa: BURSA.ID_BURSA
#  - cod_moneda: MONEDA.COD_MONEDA
#  - denumire_simbol: String
#  - sector: String
# 
# Tabel 2: istoric_pret
#  - ticker #1
#  - data_cotatie #2
#  - pret_deschidere
#  - pret_inchidere
#  - pret_maxim
#  - pret_minim
#  - volum
# 
# Cerinta 11: Crearea tabelelor în SQL și inserarea de date coerente în fiecare dintre acestea: 
#    - minimum 5 înregistrări în fiecare tabel neasociativ; 
#    - minimum 10 înregistrări în tabelele asociative; 
#    - maxim 30 de înregistrări în fiecare tabel 

# Companii pentru care extrag date de la bursa folosind yahoo finance
class DownloadCompanie:
   ticker_yf: str
   ticker: str
   denumire_companie: str
   cod_bursa: str
   cod_moneda: str
   sector: str
   
   def __init__(self, ticker_yf: str, ticker: str, denumire_companie: str, cod_bursa: str, cod_moneda: str, sector: str):
      self.ticker_yf = ticker_yf
      self.ticker = ticker
      self.denumire_companie = denumire_companie
      self.cod_bursa = cod_bursa
      self.cod_moneda = cod_moneda
      self.sector = sector
   

target_companies: list[DownloadCompanie] = [
   DownloadCompanie(ticker_yf='TLV.RO', ticker='TLV', denumire_companie='Banca Transilvania', cod_bursa='BVB', cod_moneda='RON', sector='Financiar'),
   DownloadCompanie(ticker_yf='NVDA', ticker='NVDA', denumire_companie='NVIDIA', cod_bursa='NASDAQ', cod_moneda='USD', sector='Tehnologie'),
   DownloadCompanie(ticker_yf='ALV.DE', ticker='ALV', denumire_companie='Allianz Group', cod_bursa='FSE', cod_moneda='EUR', sector='Asigurari'),
   DownloadCompanie(ticker_yf='NVO', ticker='NVO', denumire_companie='Novo Nordisk', cod_bursa='NYSE', cod_moneda='USD', sector='Sanatate'),
   DownloadCompanie(ticker_yf='AAPL', ticker='AAPL', denumire_companie='Apple', cod_bursa='NASDAQ', cod_moneda='USD', sector='Tehnologie'),
]

# Create SQL insert statements for SIMBOL_BURSIER table based on target_companies
simbol_bursier_sql_inserts: list[str] = []
for comp in target_companies:
   print(f"\n[simbol_bursier] Generate sql insert pentru \"{comp.ticker}\" - {comp.cod_bursa} (\"{comp.cod_moneda}\") ...")
   
   # Obtine ID_COMPANIE (pkey) din tabela COMPANIE folosind DENUMIRE 
   id_companie: int = db_run_sql(conn=oracle_conn, 
      sql="SELECT id_companie FROM companie WHERE denumire = :denumire", params={'denumire': comp.denumire_companie}, 
      fetch=True)[0][0] # type: ignore 

   # Obtine ID_BURSA (pkey) din tabela BURSA folosind COD_BURSA
   id_bursa: int = db_run_sql(conn=oracle_conn,
      sql="SELECT id_bursa FROM bursa WHERE cod_bursa = :cod_bursa", params={'cod_bursa': comp.cod_bursa},
      fetch=True)[0][0] # type: ignore
   
   insert_sql = f"""
      INSERT INTO simbol_bursier (ticker, id_companie, id_bursa, cod_moneda, denumire_simbol, sector) 
      VALUES ('{comp.ticker}', {id_companie}, {id_bursa}, '{comp.cod_moneda}', '{comp.denumire_companie}', '{comp.sector}');
   """
   insert_sql = re.sub(r'\s+', ' ', insert_sql).strip()
   simbol_bursier_sql_inserts.append(insert_sql)

# Create SQL insert statements for ISTORIC_PRET table
istoric_pret_sql_inserts: list[str] = []
for comp in target_companies: 
   print(f"\n[istoric_pret] Descarcare date pentru \"{comp.ticker}\" - {comp.cod_bursa} (\"{comp.cod_moneda}\") ...")
   stock_prices: list[StockPrice] = download_ticker(
      ticker=comp.ticker_yf,
      id_bursa=comp.cod_bursa,
      start_date='2026-04-01',
      end_date=datetime.now().strftime('%Y-%m-%d'),
      interval='1d', show_graph=False
   )
   print(f"Descarcat {len(stock_prices)} preturi pentru \"{comp.ticker}\" - {comp.cod_bursa} (\"{comp.cod_moneda}\")")
   
   for sp in stock_prices:
      insert_sql = f"""
         INSERT INTO istoric_pret (ticker, data_cotatie, pret_deschidere, pret_inchidere, pret_maxim, pret_minim, volum)
         VALUES ('{comp.ticker}', TO_TIMESTAMP('{sp.data_cotatie}', 'YYYY-MM-DD HH24:MI:SS'), {sp.pret_deschidere:.3f}, {sp.pret_inchidere:.3f}, {sp.pret_maxim:.3f}, {sp.pret_minim:.3f}, {sp.volum});
      """
      insert_sql = re.sub(r'\s+', ' ', insert_sql).strip()
      istoric_pret_sql_inserts.append(insert_sql)

# Shuffle istoric_pret_sql_inserts
np.random.shuffle(istoric_pret_sql_inserts)
if len(istoric_pret_sql_inserts) > 30:
   istoric_pret_sql_inserts = istoric_pret_sql_inserts[:30]

In [ ]:
for l in simbol_bursier_sql_inserts:
   print(l)
print('--------------------------------------------------------')
for l in istoric_pret_sql_inserts:
   print(l)